# Домашнее задание 2 - Титаник

Анализ датасета о пассажирах Титаника

In [1]:
import pandas as pd
import re

df = pd.read_csv('train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Основная информация о датасете

Посмотрим на типы данных, пропуски и описательную статистику

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [3]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [4]:
# пропуски по столбцам
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


В датасете 891 запись и 12 столбцов. Числовые столбцы: PassengerId, Survived, Pclass, Age, SibSp, Parch, Fare. Остальные - категориальные или текстовые

Больше всего пропусков в столбце Cabin. В Age тоже есть пропуски. В Embarked пропущено всего 2 значения

## Процент выживаемости по классам

Группируем по Pclass и считаем среднее от Survived (0 не выжил, 1 выжил), умножаем на 100

In [5]:
surv = df.groupby('Pclass')['Survived'].mean() * 100
surv.round(2)

,Survived
Pclass,
1,62.96
2,47.28
3,24.24


Пассажиры первого класса выживали значительно чаще - около 63%, второго - около 47%, третьего - только около 24%

## Самое популярное мужское и женское имя

В колонке Name формат: "Фамилия, Обращение. Имя ..." . У замужних женщин (Mrs.) настоящее имя часто указано в скобках, а перед скобками имя мужа. Поэтому для Mrs. берём имя из скобок, для остальных - первое слово после обращения.

In [6]:
def get_name(s):
    after = s.split(', ', 1)[1]
    # если есть скобки — настоящее имя там
    if '(' in after:
        m = re.search(r'\((\w+)', after)
        if m:
            return m.group(1)
    # иначе первое слово после обращения
    m = re.search(r'\w+\.\s+(\w+)', after)
    if m:
        return m.group(1)
    return None

df['FirstName'] = df['Name'].apply(get_name)

men = df[df['Sex'] == 'male']
women = df[df['Sex'] == 'female']

top_m = men['FirstName'].value_counts().idxmax()
top_w = women['FirstName'].value_counts().idxmax()

print(f'Самое популярное мужское имя: {top_m} ({men["FirstName"].value_counts().max()} человек)')
print(f'Самое популярное женское имя: {top_w} ({women["FirstName"].value_counts().max()} человек)')

Самое популярное мужское имя: William (35 человек)
Самое популярное женское имя: Anna (15 человек)


## Популярные имена по классам

То же самое, но для каждого класса отдельно

In [7]:
for cl in sorted(df['Pclass'].unique()):
    cl_df = df[df['Pclass'] == cl]
    m = cl_df[cl_df['Sex'] == 'male']['FirstName'].value_counts().idxmax()
    w = cl_df[cl_df['Sex'] == 'female']['FirstName'].value_counts().idxmax()
    print(f'Класс {cl}: мужское - {m}, женское - {w}')

Класс 1: мужское - William, женское - Elizabeth
Класс 2: мужское - William, женское - Elizabeth
Класс 3: мужское - William, женское - Anna


## Пассажиры старше 44 лет

In [8]:
old = df[df['Age'] > 44]
print(f'Всего пассажиров старше 44: {len(old)}')
old

Всего пассажиров старше 44: 115


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FirstName
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S,Timothy
11,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S,Elizabeth
15,16,1,2,"Hewlett, Mrs. (Mary D Kingcome)",female,55.0,0,0,248706,16.0000,NaN,S,Mary
33,34,0,2,"Wheadon, Mr. Edward H",male,66.0,0,0,C.A. 24579,10.5000,NaN,S,Edward
52,53,1,1,"Harper, Mrs. Henry Sleeper (Myna Haxtun)",female,49.0,1,0,PC 17572,76.7292,D33,C,Myna
...,...,...,...,...,...,...,...,...,...,...,...,...,...
857,858,1,1,"Daly, Mr. Peter Denis",male,51.0,0,0,113055,26.5500,E17,S,Peter
862,863,1,1,"Swift, Mrs. Frederick Joel (Margaret Welles Ba...",female,48.0,0,0,17466,25.9292,D17,S,Margaret
871,872,1,1,"Beckwith, Mrs. Richard Leonard (Sallie Monypeny)",female,47.0,1,1,11751,52.5542,D35,S,Sallie
873,874,0,3,"Vander Cruyssen, Mr. Victor",male,47.0,0,0,345765,9.0000,NaN,S,Victor


## Мужчины младше 44 лет

Фильтруем по двум условиям: возраст < 44 и пол = мужской

In [9]:
young_men = df[(df['Age'] < 44) & (df['Sex'] == 'male')]
print(f'Всего таких пассажиров: {len(young_men)}')
young_men

Всего таких пассажиров: 368


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FirstName
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.250,NaN,S,Owen
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.050,NaN,S,William
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.075,NaN,S,Gosta
12,13,0,3,"Saundercock, Mr. William Henry",male,20.0,0,0,A/5. 2151,8.050,NaN,S,William
13,14,0,3,"Andersson, Mr. Anders Johan",male,39.0,1,5,347082,31.275,NaN,S,Anders
...,...,...,...,...,...,...,...,...,...,...,...,...,...
883,884,0,2,"Banfield, Mr. Frederick James",male,28.0,0,0,C.A./SOTON 34068,10.500,NaN,S,Frederick
884,885,0,3,"Sutehall, Mr. Henry Jr",male,25.0,0,0,SOTON/OQ 392076,7.050,NaN,S,Henry
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.000,NaN,S,Juozas
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.000,C148,C,Karl


## N-местные кабины

У некоторых пассажиров в столбце Cabin указано несколько кабин через пробел (C23 C25 C27). Разобьём такие записи на отдельные кабины, затем посчитаем сколько пассажиров приходится на каждую кабину, и сгруппируем по количеству жильцов.

In [10]:
# берем только строки с заполненной кабиной
with_cabin = df.dropna(subset=['Cabin'])

# разбиваем составные кабины на отдельные
cabins = with_cabin['Cabin'].str.split(' ').explode()

# считаем сколько человек в каждой кабине
people_per_cabin = cabins.value_counts()

# группируем: сколько кабин с N жильцами
res = people_per_cabin.value_counts().sort_index()
res.index.name = 'Человек в кабине'
res.name = 'Количество кабин'
print(res)

Человек в кабине
1    104
2     44
3      6
4      7
Name: Количество кабин, dtype: int64


Большинство кабин одноместные. Встречаются также 2, 3 и 4 местные кабины

## Пассажиры без родственников на борту

SibSp - количество братьев/сестёр/супругов, Parch - количество родителей/детей. Если оба равны 0, значит пассажир был один.

In [11]:
alone = df[(df['SibSp'] == 0) & (df['Parch'] == 0)]
print(f'Пассажиров без родственников на борту: {len(alone)}')

Пассажиров без родственников на борту: 537


Большая часть пассажиров путешествовали без родственников на борту.